# 03: Probing Experiments — Within-Relation Truth Probe

**Task**: for each spatial relation with at least 50 training examples, restrict to the examples of that relation and train a logistic regression probe on frozen embeddings to predict VSR's True/False caption label.

**Main setting**
- Probe: `StandardScaler` (fit on each training fold) followed by `LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42)`
- Cross-validation: `StratifiedGroupKFold(5, shuffle=True, random_state=42)`. Groups are captions for SBERT and CLIP Text, image files for CLIP Image, and joint caption-or-image connected components for CLIP Concat, so no caption (or image) is in both the training and the test fold.
- Chance reference: a label-permutation test (1,000 shuffles of the True/False labels within each relation, full cross-validation rerun each time), produced by `permutation_test.py`.
- Metric: macro F1. `min_samples=50` (36 relations).

Chosen after the regularization sweep in notebook 04 had been run and seen. Reasons given: grouped folds because under stratified folds 34% of test rows have their caption in training only with the opposite label; `StandardScaler` because without it some probes predict a single class for a whole relation; `C=1.0` because it is the scikit-learn default. The setting lives in the `C`, `STANDARDIZE`, `N_SPLITS` and `MIN_SAMPLES` constants in the Setup cell; changing it also requires rerunning `permutation_test.py` with matching flags (section 5 checks that they match).

**Secondary**: the same probe under plain `StratifiedKFold(5, shuffle=True, random_state=42)` is saved alongside for reference.

Other settings (`C` from 0.01 to 1000, with and without scaling) and the check of repeated captions across stratified folds are in notebook 04 and `results/diagnostics/`.

Before running this notebook, run:

```
python permutation_test.py --standardize --C 1.0 --cv grouped --models sbert clip_text clip_image clip_concat
```

**Correction note**: an earlier version of this notebook fit a one-vs-rest relation-identity probe on all examples (`probe_relation_identity`, formerly `probe_by_relation_type`). Those results are kept in `results/relation_identity/`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os

# Notebooks run from notebooks/ — resolve paths relative to project root
PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CACHE_DIR = Path(os.getenv("CACHE_DIR", "results/embeddings"))
if not CACHE_DIR.is_absolute():
    CACHE_DIR = PROJECT_ROOT / CACHE_DIR
OUT_DIR = PROJECT_ROOT / "results" / "truth_probe"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# MAIN PROBE SETTING — the single place it is defined for this notebook.
# Folds: StratifiedGroupKFold (grouped) is the main result; stratified folds are secondary.
# If you change C or STANDARDIZE, rerun permutation_test.py with the same values first.
C = 1.0
STANDARDIZE = True
N_SPLITS = 5
MIN_SAMPLES = 50
LARGE_N = 150      # threshold for the "n >= 150" summary subset
TOL = 1e-9         # tie tolerance

np.random.seed(42)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUT_DIR}")

Project root: /Users/norair/Documents/Documents - Nora’s MacBook Air (2)/Code/Projects/spatial_probing
Output directory: /Users/norair/Documents/Documents - Nora’s MacBook Air (2)/Code/Projects/spatial_probing/results/truth_probe


## 2. Load Embeddings and Dataset Labels

In [2]:
from datasets import load_dataset

MODELS = ["sbert", "clip_text", "clip_image", "clip_concat"]
DISPLAY = {"sbert": "SBERT", "clip_text": "CLIP Text", "clip_image": "CLIP Image", "clip_concat": "CLIP Concat"}

embeddings = {m: np.load(CACHE_DIR / f"{m}_vsr_train.npy") for m in MODELS}
for m, e in embeddings.items():
    print(f"  {m:12s} {e.shape}")

train_data = load_dataset("cambridgeltl/vsr_random")["train"]
relation_types = np.array(train_data["relation"])
binary_labels = np.array([int(v) for v in train_data["label"]])
captions = np.array(train_data["caption"])
image_files = np.array(train_data["image"])

for e in embeddings.values():
    assert e.shape[0] == len(train_data)

print(f"\nDataset: {len(train_data)} examples")
print(f"Relation types: {len(np.unique(relation_types))} unique")
print(f"Labels: {binary_labels.sum()} True, {len(binary_labels) - binary_labels.sum()} False")
print(f"Unique captions: {len(np.unique(captions))}, unique images: {len(np.unique(image_files))}")

  sbert        (7680, 768)
  clip_text    (7680, 512)
  clip_image   (7680, 512)
  clip_concat  (7680, 1024)



Dataset: 7680 examples
Relation types: 64 unique
Labels: 3876 True, 3804 False
Unique captions: 3762, unique images: 4968


## 3. Train Probes

`results` holds the main setting (grouped folds); `results_stratified` holds the secondary run (plain stratified folds).

In [3]:
from src.probing import probe_truth_by_relation

GROUPS = {
    "sbert": captions,
    "clip_text": captions,
    "clip_image": image_files,
    "clip_concat": [captions, image_files],   # joint: shared caption OR shared image
}
GROUP_DESC = {"sbert": "caption", "clip_text": "caption", "clip_image": "image", "clip_concat": "caption+image (joint)"}

results = {}             # main: grouped folds
results_stratified = {}  # secondary: stratified folds
for m in MODELS:
    results[m] = probe_truth_by_relation(
        embeddings[m], relation_types, binary_labels, n_splits=N_SPLITS, C=C, min_samples=MIN_SAMPLES,
        groups=GROUPS[m], standardize=STANDARDIZE,
    )
    results_stratified[m] = probe_truth_by_relation(
        embeddings[m], relation_types, binary_labels, n_splits=N_SPLITS, C=C, min_samples=MIN_SAMPLES,
        standardize=STANDARDIZE,
    )
    print(f"{m:12s} relations: {len(results[m])}  "
          f"mean macro F1 grouped by {GROUP_DESC[m]}: {results[m]['f1_macro_mean'].mean():.4f}  "
          f"stratified: {results_stratified[m]['f1_macro_mean'].mean():.4f}")

sbert        relations: 36  mean macro F1 grouped by caption: 0.4870  stratified: 0.3080


clip_text    relations: 36  mean macro F1 grouped by caption: 0.4889  stratified: 0.3041


clip_image   relations: 36  mean macro F1 grouped by image: 0.5426  stratified: 0.5386


clip_concat  relations: 36  mean macro F1 grouped by caption+image (joint): 0.5527  stratified: 0.4779


## 4. Group Sizes per Relation

Number of groups and largest group per relation, for each grouping.

In [4]:
gs = results["sbert"][["relation", "n_samples"]].copy()
for m, name in [("sbert", "caption"), ("clip_image", "image"), ("clip_concat", "joint")]:
    g = results[m].set_index("relation")
    gs[f"{name}_groups"] = gs["relation"].map(g["n_groups"])
    gs[f"{name}_max"] = gs["relation"].map(g["max_group_size"])
    gs[f"{name}_max_pct"] = (100 * gs[f"{name}_max"] / gs["n_samples"]).round(1)
gs.to_csv(OUT_DIR / "group_sizes.csv", index=False)
print(gs.to_string(index=False))
print("\nLargest group as % of relation:",
      {n: float(gs[f"{n}_max_pct"].max()) for n in ["caption", "image", "joint"]})

            relation  n_samples  caption_groups  caption_max  caption_max_pct  image_groups  image_max  image_max_pct  joint_groups  joint_max  joint_max_pct
            touching        884             268           22              2.5           806          3            0.3           201         60            6.8
         in front of        528             271            9              1.7           515          2            0.4           258         22            4.2
              behind        498             251           12              2.4           487          2            0.4           240         15            3.0
               under        415             151           15              3.6           406          2            0.5           143         24            5.8
                  on        408             159           13              3.2           398          2            0.5           149         17            4.2
           on top of        349             138     

## 5. Permutation Test Results

Read from `results/truth_probe/permutation_test.csv` (written by `permutation_test.py`). For each relation, `null_mean` is the mean macro F1 over 1,000 label shuffles and `p_upper = (1 + #{null >= observed}) / 1001`. The cell checks that the file was produced at this notebook's setting and that its observed scores equal the ones computed above.

In [5]:
perm = pd.read_csv(OUT_DIR / "permutation_test.csv")
perm_summary = pd.read_csv(OUT_DIR / "permutation_test_summary.csv")

assert set(perm["cv"]) == {"grouped"}, "permutation test must be run with --cv grouped"
assert np.allclose(perm["C"], C) and (perm["standardize"] == STANDARDIZE).all(), "permutation test setting differs"
assert set(MODELS) <= set(perm["condition"]), "permutation test is missing a condition"
for m in MODELS:
    p = perm[perm["condition"] == m].set_index("relation")
    obs = results[m].set_index("relation")["f1_macro_mean"]
    assert np.abs(p["observed_f1"].reindex(obs.index) - obs).max() < 1e-9, f"{m}: observed F1 mismatch"

N_PERM = int(perm["n_permutations"].iloc[0])
print(f"Permutation test: {N_PERM} permutations, C={C}, standardize={STANDARDIZE}, grouped folds; observed scores match.")
print(perm_summary.drop(columns=["cv", "C", "standardize", "n_permutations"]).round(4).to_string(index=False))

Permutation test: 1000 permutations, C=1.0, standardize=True, grouped folds; observed scores match.
  condition              grouping     relations  n_relations  observed_mean_f1  null_mean_f1  null_std_of_mean  null_p2.5_of_mean  null_p97.5_of_mean  p_upper_of_mean  p_lower_of_mean  n_relations_p_upper_lt_0.05  n_relations_p_upper_lt_0.05_holm  n_relations_p_lower_lt_0.05  n_relations_p_lower_lt_0.05_holm
      sbert               caption all (n >= 50)           36            0.4870        0.4816            0.0097             0.4635              0.5002           0.2877           0.7133                            0                                 0                            0                                 0
      sbert               caption      n >= 150           19            0.4929        0.4882            0.0088             0.4708              0.5050           0.3037           0.6973                            0                                 0                            0     

## 6. Comparison Table

Per relation, sorted by `n_samples`: macro F1 under the main setting (`<model>_f1`), the permutation null mean (`<model>_null_mean`) and one-sided p-value (`<model>_p_upper`), the `most_frequent` dummy baseline, and macro F1 under stratified folds (`<model>_f1_stratified`).

In [6]:
comparison = results["sbert"][["relation", "n_samples", "n_true", "n_false"]].copy()
for m in MODELS:
    comparison[f"{m}_f1"] = comparison["relation"].map(results[m].set_index("relation")["f1_macro_mean"])
for m in MODELS:
    p = perm[perm["condition"] == m].set_index("relation")
    comparison[f"{m}_null_mean"] = comparison["relation"].map(p["null_mean"])
    comparison[f"{m}_p_upper"] = comparison["relation"].map(p["p_upper"])
comparison["baseline_most_frequent_f1"] = comparison["relation"].map(
    results_stratified["sbert"].set_index("relation")["baseline_most_frequent_f1_macro"]
)
for m in MODELS:
    comparison[f"{m}_f1_stratified"] = comparison["relation"].map(
        results_stratified[m].set_index("relation")["f1_macro_mean"]
    )

comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
print(comparison[["relation", "n_samples"] + [f"{m}_f1" for m in MODELS] + [f"{m}_p_upper" for m in MODELS]]
      .round(3).to_string(index=False))

            relation  n_samples  sbert_f1  clip_text_f1  clip_image_f1  clip_concat_f1  sbert_p_upper  clip_text_p_upper  clip_image_p_upper  clip_concat_p_upper
            touching        884     0.505         0.514          0.605           0.603          0.321              0.168               0.001                0.001
         in front of        528     0.509         0.507          0.514           0.527          0.318              0.325               0.279                0.134
              behind        498     0.516         0.506          0.538           0.517          0.217              0.336               0.081                0.251
               under        415     0.474         0.481          0.550           0.539          0.715              0.644               0.040                0.068
                  on        408     0.503         0.510          0.569           0.590          0.363              0.288               0.014                0.001
           on top of        

## 7. Summary Statistics

`std across relations` is the standard deviation of the per-relation mean macro F1. `mean fold std` is the average, over relations, of the standard deviation across the 5 folds. For the main setting, the `null_*` and `p_*` columns come from the permutation test: `null_mean_f1` and the 2.5/97.5 percentiles describe the null distribution of the across-relation mean (the same shuffle index is used for every relation), and `n_p_lt_0.05` counts relations with `p_upper < 0.05` (`_holm`: after Holm correction across the relations in the row). Rows are given for all 36 relations and for the relations with n >= 150.

In [7]:
def describe(f1, fold_std):
    return {
        "mean_f1": f1.mean(),
        "std across relations": f1.std(ddof=0),
        "median_f1": f1.median(),
        "min_f1": f1.min(),
        "max_f1": f1.max(),
        "mean fold std": fold_std.mean(),
        "n_relations": len(f1),
    }

rows = []
for subset_label, min_n in [(f"all (n >= {MIN_SAMPLES})", MIN_SAMPLES), (f"n >= {LARGE_N}", LARGE_N)]:
    for m in MODELS:
        r = results[m][results[m]["n_samples"] >= min_n]
        ps = perm_summary[(perm_summary["condition"] == m) & (perm_summary["relations"] == subset_label)].iloc[0]
        assert abs(ps["observed_mean_f1"] - r["f1_macro_mean"].mean()) < 1e-9
        rows.append({
            "condition": DISPLAY[m], "cv": f"grouped ({GROUP_DESC[m]})", "relations": subset_label,
            **describe(r["f1_macro_mean"], r["f1_macro_std"]),
            "null_mean_f1": ps["null_mean_f1"],
            "null_p2.5": ps["null_p2.5_of_mean"],
            "null_p97.5": ps["null_p97.5_of_mean"],
            "p_upper_of_mean": ps["p_upper_of_mean"],
            "n_p_lt_0.05": int(ps["n_relations_p_upper_lt_0.05"]),
            "n_p_lt_0.05_holm": int(ps["n_relations_p_upper_lt_0.05_holm"]),
            "baseline_most_frequent_mean_f1": r["baseline_most_frequent_f1_macro"].mean(),
        })
    for m in MODELS:
        r = results_stratified[m][results_stratified[m]["n_samples"] >= min_n]
        rows.append({
            "condition": DISPLAY[m], "cv": "stratified", "relations": subset_label,
            **describe(r["f1_macro_mean"], r["f1_macro_std"]),
            "baseline_most_frequent_mean_f1": r["baseline_most_frequent_f1_macro"].mean(),
        })

summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary_statistics.csv", index=False)
print(summary.round(4).to_string(index=False))

  condition                              cv     relations  mean_f1  std across relations  median_f1  min_f1  max_f1  mean fold std  n_relations  null_mean_f1  null_p2.5  null_p97.5  p_upper_of_mean  n_p_lt_0.05  n_p_lt_0.05_holm  baseline_most_frequent_mean_f1
      SBERT               grouped (caption) all (n >= 50)   0.4870                0.0397     0.4889  0.3862  0.5919         0.0543           36        0.4816     0.4635      0.5002           0.2877          0.0               0.0                          0.3499
  CLIP Text               grouped (caption) all (n >= 50)   0.4889                0.0431     0.4808  0.4104  0.5891         0.0564           36        0.4816     0.4628      0.5011           0.2338          0.0               0.0                          0.3499
 CLIP Image                 grouped (image) all (n >= 50)   0.5426                0.0833     0.5304  0.3736  0.7010         0.0686           36        0.4919     0.4719      0.5107           0.0010         12.0       

## 8. Wins / Ties / Losses

Per-relation macro F1 compared for each pair of conditions. A tie is `abs(a - b) < 1e-9`. "a_wins" counts relations where `a` is higher.

In [8]:
from itertools import combinations

def wins_ties_losses(a: pd.Series, b: pd.Series) -> tuple[int, int, int]:
    diff = (a - b).to_numpy()
    ties = int((np.abs(diff) < TOL).sum())
    wins = int((diff >= TOL).sum())
    losses = int((diff <= -TOL).sum())
    return wins, ties, losses

wtl_rows = []
for cv_name, res in [("grouped", results), ("stratified", results_stratified)]:
    for subset_label, min_n in [(f"all (n >= {MIN_SAMPLES})", MIN_SAMPLES), (f"n >= {LARGE_N}", LARGE_N)]:
        f1 = {m: res[m][res[m]["n_samples"] >= min_n].set_index("relation")["f1_macro_mean"] for m in MODELS}
        for a, b in combinations(MODELS, 2):
            fb = f1[b].reindex(f1[a].index)
            w, t, l = wins_ties_losses(f1[a], fb)
            wtl_rows.append({"cv": cv_name, "relations": subset_label, "a": a, "b": b,
                             "a_wins": w, "ties": t, "a_losses": l,
                             "mean_diff_a_minus_b": (f1[a] - fb).mean()})

wtl = pd.DataFrame(wtl_rows)
wtl.to_csv(OUT_DIR / "wins_ties_losses.csv", index=False)
print(wtl.round(4).to_string(index=False))

        cv     relations          a           b  a_wins  ties  a_losses  mean_diff_a_minus_b
   grouped all (n >= 50)      sbert   clip_text      17     0        19              -0.0019
   grouped all (n >= 50)      sbert  clip_image      11     0        25              -0.0556
   grouped all (n >= 50)      sbert clip_concat       7     0        29              -0.0657
   grouped all (n >= 50)  clip_text  clip_image      10     0        26              -0.0537
   grouped all (n >= 50)  clip_text clip_concat       6     0        30              -0.0639
   grouped all (n >= 50) clip_image clip_concat      13     0        23              -0.0102
   grouped      n >= 150      sbert   clip_text       8     0        11              -0.0018
   grouped      n >= 150      sbert  clip_image       3     0        16              -0.0800
   grouped      n >= 150      sbert clip_concat       1     0        18              -0.0856
   grouped      n >= 150  clip_text  clip_image       4     0        1

## 9. Save Detailed Results

Per-relation results for each condition and fold scheme, including fold std (`f1_macro_std`, `accuracy_std`), precision, recall and the two single-seed dummy baselines.

In [9]:
for m in MODELS:
    results[m].to_csv(OUT_DIR / f"{m}_grouped_detailed.csv", index=False)
    results_stratified[m].to_csv(OUT_DIR / f"{m}_stratified_detailed.csv", index=False)
    print(f"Saved: {m}_grouped_detailed.csv, {m}_stratified_detailed.csv")

print("\nFiles in results/truth_probe/:")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {f.name}")

Saved: sbert_grouped_detailed.csv, sbert_stratified_detailed.csv
Saved: clip_text_grouped_detailed.csv, clip_text_stratified_detailed.csv
Saved: clip_image_grouped_detailed.csv, clip_image_stratified_detailed.csv
Saved: clip_concat_grouped_detailed.csv, clip_concat_stratified_detailed.csv

Files in results/truth_probe/:
  clip_concat_grouped_detailed.csv
  clip_concat_stratified_detailed.csv
  clip_image_grouped_detailed.csv
  clip_image_stratified_detailed.csv
  clip_text_grouped_detailed.csv
  clip_text_stratified_detailed.csv
  comparison.csv
  group_sizes.csv
  permutation_test.csv
  permutation_test_summary.csv
  sbert_grouped_detailed.csv
  sbert_stratified_detailed.csv
  summary_statistics.csv
  wins_ties_losses.csv
